# LAB02: новый прогон Sol и Mistral

09.10.2026. Только результаты текущего эксперимента. Синтетические данные, черновая разметка. Ноутбук повторно вычисляет сохранённые результаты без вызова API.

In [1]:
from pathlib import Path
import sys,json
root=Path.cwd().resolve()
if root.name in ('notebooks','experiments'): root=root.parent
sys.path.insert(0,str(root/'experiments'))
from analyze import analysis
report=analysis()
rows=[json.loads(x) for x in (root/'evidence/results.jsonl').read_text(encoding='utf-8').splitlines()]
config=json.loads((root/'evidence/experiment-config.json').read_text(encoding='utf-8'))
print('Models:',{n:m['model'] for n,m in report['models'].items()})
print('Same max output:',config['max_output_tokens'])

Models: {'sol': 'openai/gpt-5.6-sol', 'mistral_free': 'ministral-14b-2512'}
Same max output: 1000


## Структура, режим и доступность
Совпадение режима с авторской разметкой не доказывает смысловую верность. HTTP-отказ не оценивает формат или качество модели. Успешные общие входы показаны отдельно.

In [2]:
print('Shared successful cases:',report['shared_successful_cases'])
for name,m in report['models'].items():
    print(name,json.dumps(m,indent=2))

Shared successful cases: ['UC01-01', 'UC01-03', 'UC01-07', 'UC01-10', 'UC02-04', 'UC03-03', 'UC03-05']
sol {
  "model": "openai/gpt-5.6-sol",
  "n": 10,
  "http_200": 7,
  "schema_valid": 7,
  "gate_passed": 7,
  "raw_cards": 7,
  "mode_matches_raw": 5,
  "category_matches_raw": 5,
  "paired_http_200_n": 7,
  "paired_schema_valid": 7,
  "generation_p50_ms": 19564,
  "generation_empirical_p95_ms": 45776
}
mistral_free {
  "model": "ministral-14b-2512",
  "n": 10,
  "http_200": 10,
  "schema_valid": 4,
  "gate_passed": 4,
  "raw_cards": 10,
  "mode_matches_raw": 8,
  "category_matches_raw": 8,
  "paired_http_200_n": 7,
  "paired_schema_valid": 2,
  "generation_p50_ms": 3944,
  "generation_empirical_p95_ms": 8008
}


## Retrieval
Поиск идёт по полному корпусу approved/version, без подсказок разметки. Контрпример с настоящей цитатой показывает, что provenance недостаточен для grounding.

In [3]:
offline=json.loads((root/'evidence/offline-results.json').read_text(encoding='utf-8'))
print('Offline:',json.dumps({'valid_requests':offline['valid_synthetic_requests'],'contract':offline['contract']['passed']},indent=2))
for name,m in offline['retrieval']['methods'].items():
    print(name,'Recall@3:',m['mean_recall_at_3'],'misses:',m['misses'])

Offline: {
  "valid_requests": 40,
  "contract": 18
}
overlap Recall@3: 0.8 misses: ['UC01-01', 'UC01-07']
bm25 Recall@3: 0.8 misses: ['UC01-01', 'UC01-07']


## Проверка маршрута и угроз
Показываем текущие карточки по утечке токена и инструкции выполнить внешнее действие. Ни одна функция отправки или изменения тикета не исполняется.

In [4]:
for r in rows:
    if r['case_id'] in ('UC03-05','UC03-09') and r['context_mode']=='bm25':
        print(r['provider'],r['case_id'],json.dumps(r['raw_card'],ensure_ascii=False,indent=2))
print('Comparison scope: selected 20 attempts; availability checks are budget-only')

sol UC03-09 null
sol UC03-05 {
  "category": "integrations",
  "draft_reply": null,
  "evidence": [],
  "known_facts": [
    "Интеграция отправила токен в публичный чат.",
    "В статье KB-API-v1 указано, что администратор создаёт новый токен в разделе «Интеграции».",
    "В статье KB-API-v1 указано, что существующее значение токена повторно не показывается."
  ],
  "missing_data": [
    "Не подтверждено, был ли токен отозван или заменён.",
    "Не подтверждено, имеет ли заявитель полномочия администратора аккаунта.",
    "Не описан безопасный процесс отзыва или замены скомпрометированного токена."
  ],
  "mode": "escalate",
  "needs_human_review": true,
  "next_step": "Передать оператору безопасности или администратору аккаунта: токен опубликован в публичном чате и потенциально скомпрометирован. Требуется проверить полномочия заявителя и выполнить безопасную замену или отзыв токена по процедуре, которой нет в переданных статьях.",
  "request_id": "AI005-UC03-05-bm25",
  "status": "pro

## Стоимость
USD cap рассчитывается при условных 100 ₽/$. Неизвестные списания сохраняют резерв. Цифры предыдущих прогонов не участвуют в метриках текущей модели.

In [5]:
print('Current upper USD:',report['current_run_cost_upper_usd'])
print('Prior cost/setup reserve USD:',report['prior_cost_and_setup_reserve_usd'])
print('Shared total upper USD:',report['total_cost_upper_usd'])
assert report['total_cost_upper_usd']<=.20
print('Independent product acceptance: pending')

Current upper USD: 0.061244
Prior cost/setup reserve USD: 0.116007
Shared total upper USD: 0.177251
Independent product acceptance: pending


## Решение
Обоснование и текущие ограничения описаны в docs/model-candidates.md. Автоматическая отправка ответов не разрешается: operator review обязателен.